# Main Cloud Orchestrator - Colab & Kaggle

This notebook serves as the main orchestrator for running experiments on Google Colab or Kaggle. It performs a sparse checkout to download only the lightweight code directory (`experiments/`), installs the package in editable mode, and executes python modules in the notebook space to preserve memory and variables for debugging.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'

# 1. Clonación superficial (shallow) y dispersa (sparse)
if not os.path.exists(REPO_NAME):
    print(f"Clonando {REPO_NAME} (solo directorio 'experiments' y sin historial)...")
    # --depth 1: Descarga solo el último commit (ignora el historial).
    # --sparse: Inicializa el modo disperso desde la clonación.
    !git clone -q --depth 1 --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Actualizando repositorio {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull -q
    %cd experiments

# 2. Verificación del directorio de trabajo
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Fallo al navegar al directorio. Ruta actual: {current_dir}")

# 3. Instalación de dependencias
print("Instalando el paquete en modo editable con dependencias [cloud]...")
%pip install -q -e .[cloud]

Clonando ia_article (solo directorio 'experiments' y sin historial)...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 2.59 KiB | 2.59 MiB/s, done.
/content/ia_article
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 595.38 KiB | 14.88 MiB/s, done.
/content/ia_article/experiments
Instalando el paquete en modo editable con dependencias [cloud]...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.0 MB/s eta 0:00:00
 

In [ ]:
# Cell 2: Mount Google Drive to load/save datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 3: Run colocated unit tests with pytest
!pytest src/

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/ia_article/experiments
configfile: pyproject.toml
plugins: cov-7.1.0, langsmith-0.10.2, anyio-4.14.2, typeguard-4.5.2
collected 2 items                                                              

src/data_preparation/test_parser.py ..                                   [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.12.13-final-0 _______________

Name                                  Stmts   Miss Branch BrPart  Cover   Missing
---------------------------------------------------------------------------------
src/data_preparation/__init__.py          0      0      0      0   100%
src/data_preparation/parser.py          176    148     30      2    15%   73, 129-167, 172-452, 456
src/data_preparation/test_parser.py      13      0      2    

In [ ]:
# Cell 4: Run the production parser on the full train.csv dataset stored in Google Drive
# Default paths are built-in, no arguments needed!
%run src/data_preparation/parser.py

Loading annotations dataset...
Computing deterministic splits...
✓ Split metadata CSV exported to: /content/drive/MyDrive/ia_article/01_processed/split_metadata.csv
✓ Configuration YAML written to: /content/drive/MyDrive/ia_article/01_processed/smart_dataset.yaml
Preparing tasks for parallel execution...
Executing parsing and OBB corner calculation using parallel workers (2)...
Aggregating statistics...

=== Performing Strict Statistical Audit ===
Total Frames: 54262 / 54262
Total OBB Objects: 601934 / 601934
Total Unique Clips: 1088 / 1088
Empty Frames ('none'): 3394 / 3394
Frames with 1 object: 6212 / 6212
Frames with >=2 objects: 44656 / 44656
Max objects in single frame: 53 / 53

Class Distribution:
  Class 0 (auto): 481731 (80.03%) [Target: 481731]
  Class 1 (combi):  10152 ( 1.69%) [Target: 10152]
  Class 2 (microbus):   2802 ( 0.47%) [Target: 2802]
  Class 3 (minibus):  18941 ( 3.15%) [Target: 18941]
  Class 4 (omnibus):   2283 ( 0.38%) [Target: 2283]
  Class 5 (articulado):    

In [ ]:
# Cell 5: Terminate process and automatically disconnect Colab VM to save credits
print("Proceso terminado.")
from google.colab import runtime
runtime.unassign()

Proceso terminado.
